# Revision Agustin

## Segmento 1 ingreso bases

In [128]:
import pandas as pd
import re
import unicodedata
from IPython.display import display

# =========================
# RUTAS
# =========================
ruta_radicados = r"./input/Radicados.xlsx"
ruta_firmados = r"./input/Firmados.xlsx"
ruta_actividades = r"./input/ActividadesActuales.xlsx"

# =========================
# CARGA
# =========================
df_radicados_raw = pd.read_excel(ruta_radicados)
df_firmados_raw = pd.read_excel(ruta_firmados, header=None)
df_actividades_raw = pd.read_excel(ruta_actividades)

print("Base RADICADOS cargada:", df_radicados_raw.shape)
print("Base FIRMADOS cargada:", df_firmados_raw.shape)
print("Base ACTIVIDADES cargada:", df_actividades_raw.shape)

Base RADICADOS cargada: (2253, 25)
Base FIRMADOS cargada: (2271, 9)
Base ACTIVIDADES cargada: (2057, 22)


## Segmento 2 — Limpieza de títulos y visualización de bases

In [129]:
    # =========================
# FUNCIONES DE LIMPIEZA
# =========================
def quitar_tildes(texto):
    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto

def limpiar_titulo(texto):
    texto = str(texto).strip()
    texto = quitar_tildes(texto)
    texto = texto.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto

def columnas_unicas(cols):
    usadas = {}
    salida = []
    for c in cols:
        if c not in usadas:
            usadas[c] = 0
            salida.append(c)
        else:
            usadas[c] += 1
            salida.append(f"{c}_{usadas[c]}")
    return salida


# =========================
# 1. RADICADOS
# =========================
df_radicados = df_radicados_raw.copy()

columnas_radicados = [limpiar_titulo(c) for c in df_radicados.columns]
columnas_radicados = columnas_unicas(columnas_radicados)
df_radicados.columns = columnas_radicados

# quitar columnas basura unnamed
df_radicados = df_radicados.loc[:, ~df_radicados.columns.str.startswith("unnamed")]

# quitar primera fila informativa
df_radicados = df_radicados.iloc[1:].reset_index(drop=True)


# =========================
# 2. FIRMADOS
# =========================
df_firmados = df_firmados_raw.copy()

# el encabezado real está en la fila 3
nuevos_titulos_firmados = df_firmados.iloc[3].tolist()
nuevos_titulos_firmados = [limpiar_titulo(c) for c in nuevos_titulos_firmados]
nuevos_titulos_firmados = columnas_unicas(nuevos_titulos_firmados)

df_firmados = df_firmados.iloc[4:].reset_index(drop=True)
df_firmados.columns = nuevos_titulos_firmados


# =========================
# 3. ACTIVIDADES ACTUALES
# =========================
df_actividades = df_actividades_raw.copy()

# la fila 0 completa los nombres reales
fila0 = df_actividades.iloc[0]

nuevos_titulos_actividades = []
for col in df_actividades.columns:
    if str(col).startswith("Unnamed"):
        nuevos_titulos_actividades.append(fila0[col])
    else:
        # si ya tiene nombre útil, lo dejamos
        nuevos_titulos_actividades.append(col)

nuevos_titulos_actividades = [limpiar_titulo(c) for c in nuevos_titulos_actividades]
nuevos_titulos_actividades = columnas_unicas(nuevos_titulos_actividades)

df_actividades = df_actividades.iloc[1:].reset_index(drop=True)
df_actividades.columns = nuevos_titulos_actividades


# =========================
# MOSTRAR TITULOS LIMPIOS
# =========================
print("\n========== TITULOS RADICADOS ==========")
print(df_radicados.columns.tolist())
display(df_radicados.head(5))

print("\n========== TITULOS FIRMADOS ==========")
print(df_firmados.columns.tolist())
display(df_firmados.head(5))

print("\n========== TITULOS ACTIVIDADES ==========")
print(df_actividades.columns.tolist())
display(df_actividades.head(5))


========== TITULOS RADICADOS ==========
['no_doc', 'fecha_correo', 'radicado', 'tipo', 'f_radicado', 'asunto', 'titulo', 'elaboro', 'reviso', 'radico', 'of_radica', 'estado', 'medio', 'tramite', 'destinatarios', 'remitente', 'responsable', 'pre_radicado', 'tipo_asunto', 'registropor', 'ver', 'info_sae', 'estado_terminos']


,no_doc,fecha_correo,radicado,tipo,f_radicado,asunto,titulo,elaboro,reviso,radico,...,tramite,destinatarios,remitente,responsable,pre_radicado,tipo_asunto,registropor,ver,info_sae,estado_terminos
0,NaN,2026-01-02 07:44:28.317,20263000014,Memorando,2026-01-02 07:44:28.300,observaciones al proyecto de acto administrati...,N.A,Jhoanna Alexandra Montoya Muñoz,NaN,Emma Constanza Zuñiga Galvis,...,Memorando Interno,Dirección Regional Tequendama,Dirección Jurídica Ambiental (Emma Constanza Z...,Argelio David Vargas Fragoz[Dirección Regional...,NaN,Devolucion de Acto Administrativo,ezunigag,NaN,NaN,NaN
1,NaN,2026-01-02 09:22:35.480,20263000020,Memorando,2026-01-02 09:22:35.413,Respuesta al radicado 06253001741: Respuesta a...,N.A,Juan Jacobo Finkielsztein Perez,NaN,Emma Constanza Zuñiga Galvis,...,Memorando Interno,Dirección Regional Gualivá,Dirección Jurídica Ambiental (Emma Constanza Z...,Aura Daniela Leon Perez[Dirección Regional Gua...,NaN,NaN,ezunigag,NaN,NaN,NaN
2,NaN,2026-01-02 09:55:54.050,20263000021,Memorando,2026-01-02 09:55:54.013,Respuesta al radicado 20253123863: Reporte Pub...,N.A,Barbara Maria Ospina Acevedo,Juan Jacobo Finkielsztein Perez / DJA,Emma Constanza Zuñiga Galvis,...,Memorando Interno,Secretaría General,Dirección Jurídica Ambiental (Emma Constanza Z...,Yadid Jimena Ariza Diaz[Secretaría General],NaN,Respuesta al memorando,ezunigag,NaN,NaN,NaN
3,NaN,2026-01-02 11:26:30.767,20263000031,Memorando,2026-01-02 11:26:30.680,Informes Técnicos Por Acoger Meta 12.2.5- Indi...,N.A,Hernan Mauricio Medina Forero,NaN,Emma Constanza Zuñiga Galvis,...,Memorando Interno,VARIOS (15 destinos),Dirección Jurídica Ambiental (Emma Constanza Z...,Alfonso Beltran Correa[Dirección Regional Gual...,NaN,NaN,ezunigag,NaN,NaN,NaN
4,NaN,2026-01-02 11:26:30.767,20263000031,Memorando,2026-01-02 11:26:30.680,Informes Técnicos Por Acoger Meta 12.2.5- Indi...,N.A,Hernan Mauricio Medina Forero,NaN,Emma Constanza Zuñiga Galvis,...,Memorando Interno,VARIOS (15 destinos),Dirección Jurídica Ambiental (Emma Constanza Z...,Diego Javier Carrillo Betancourt[Dirección Reg...,NaN,NaN,ezunigag,NaN,NaN,NaN



========== TITULOS FIRMADOS ==========
['expediente', 'tipo_tramite', 'tramite', 'estado_31dic2025', 'etapa_actual', 'regional', 'fecha_apertura', 'ano_creacion', 'estado_31mar']


,expediente,tipo_tramite,tramite,estado_31dic2025,etapa_actual,regional,fecha_apertura,ano_creacion,estado_31mar
0,26551,Permisivo,Licencia Ambiental,En Trámite,Expedición de Acto Administrativo,Sabana Centro,2005-10-14 00:00:00,2005,Resuelto
1,29164,Permisivo,Permiso de vertimientos,En Trámite,Evaluación,Sabana Centro,2007-02-28 00:00:00,2007,Resuelto
2,33536,Sancionatorio,Afectación recurso Suelo,En Trámite,Etapa Decisoria,Sabana Centro,2009-04-16 10:38:00,2009,Resuelto
3,41221,Sancionatorio,Afectación recurso Suelo,En Trámite,Etapa de Conocimiento,Ubate,2012-08-17 15:01:01.850000,2012,Resuelto
4,47452,Permisivo,Reglamentación de corrientes de aguas de uso p...,En Trámite,Expedición de Acto Administrativo,Soacha,2014-09-12 14:48:47.307000,2014,Resuelto



========== TITULOS ACTIVIDADES ==========
['expediente', 'tipo_de_tramite', 'tramite', 'estado_actual', 'regional', 'pac', 'fecha_apertura', 'responsable', 'fecha_asignacion', 'estado_usuario', 'dias_responsable', 'formatoexp', 'etapa', 'actividad', 'sidcar', 'estado', 'devoluciones', 'radicado', 'remite', 'fecha_asignacion_1', 'dias_responsable_1', 'comentarios']


,expediente,tipo_de_tramite,tramite,estado_actual,regional,pac,fecha_apertura,responsable,fecha_asignacion,estado_usuario,...,etapa,actividad,sidcar,estado,devoluciones,radicado,remite,fecha_asignacion_1,dias_responsable_1,comentarios
0,112546,Sancionatorio,Afectación recurso Aire,En Trámite,Magdalena Centro,Actividad 12.2.1,2025-02-10 16:46:56.183000,LADY PAOLA GALINDO RAMIREZ (DJA),2026-04-07 12:59:43.260000,Activo,...,Etapa de Conocimiento,Verificación,713241,Devuelto para correcciones,1,NaN,Leidy Viviana Alvarado Torres (DJA),2026-04-03 14:33:19.663000,2,Se considera necesario revisar la norma presun...
1,71180,Sancionatorio,Afectación recurso Agua,En Trámite,Chiquinquira,Actividad 12.2.1,2018-09-21 09:26:04.033000,Angela Maria Bedoya Gonzalez (DJA),2026-04-07 12:55:28,Activo,...,Etapa Decisoria,Acto administrativo que decide de fondo,NaN,NaN,NaN,NaN,SANDRA YISEL BORBON RODRIGUEZ (DJA),2026-03-24 10:33:13,8,Buenas tards doctora Angela se devuelve el exp...
2,73575,Sancionatorio,Afectación recurso Suelo,En Trámite,Sabana Centro,Actividad 12.2.1,2019-02-07 10:22:54.580000,Lorena Del Pilar Riaño Garcia (DJA),2026-04-07 12:53:40.787000,Activo,...,Etapa Decisoria,Acto administrativo que decide de fondo,612594,Enviado para Revisión,7,NaN,Jeimy Tatiana Plata Rueda (DJA),2026-03-25 18:48:29.827000,7,Dra por favor revisar comentarios en rojo al i...
3,84798,Sancionatorio,Afectación recurso Suelo,En Trámite,Sumapaz,Actividad 12.2.1,2020-12-30 19:20:05.363000,Juan Carlos Cubillos Pineda (DJA),2026-04-07 12:50:57,Activo,...,Etapa de Conocimiento,Formulación de cargos,NaN,NaN,NaN,NaN,Rosmery Nuñez Varon (DRSU),2026-03-18 11:11:21,11,NaN
4,97257,Sancionatorio,Afectación recurso Flora,En Trámite,Sumapaz,Actividad 12.2.1,2023-01-06 15:51:11.570000,Juan Carlos Cubillos Pineda (DJA),2026-04-07 12:49:43,Activo,...,Etapa de Conocimiento,Formulación de cargos,NaN,NaN,NaN,NaN,Rosmery Nuñez Varon (DRSU),2026-04-07 12:48:27,0,NaN


# Construcción de nueva tabla desde Radicados

## Segmento 3

In [130]:
# ==========================================
# SEGMENTO 3 - FILTRO DEVOLUCIONES + EXTRACCIÓN EXPEDIENTE
# ==========================================

import pandas as pd
import re
from IPython.display import display

rad = df_radicados.copy()

# ------------------------------------------
# 1. FILTRO: todo lo relacionado con devolución
# ------------------------------------------
patron_devolucion = r"devoluc|devuelt|devuelta|devolver"

rad["tipo_asunto_texto"] = rad["tipo_asunto"].astype(str).str.lower().str.strip()

rad_devoluciones = rad[
    rad["tipo_asunto_texto"].str.contains(patron_devolucion, na=False, regex=True)
].copy()

# ------------------------------------------
# 2. EXTRACTOR ROBUSTO DE EXPEDIENTE
# ------------------------------------------
def extraer_expediente_desde_asunto(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).strip()

    patrones = [
        # expediente / expediente. / expediente:
        r'\bexpediente\b\s*[:#\-\./]?\s*([0-9]{3,})',

        # exped / exped.
        r'\bexped\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexped\.\s*([0-9]{3,})',

        # expte / expte.
        r'\bexpte\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexpte\.\s*([0-9]{3,})',

        # exp / exp.
        r'\bexp\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexp\.\s*([0-9]{3,})',

        # ex / ex.
        r'\bex\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bex\.\s*([0-9]{3,})',

        # cuando el número aparece más adelante dentro del texto
        r'\bexpediente\b.*?([0-9]{3,})',
        r'\bexped\b.*?([0-9]{3,})',
        r'\bexpte\b.*?([0-9]{3,})',
        r'\bexp\b.*?([0-9]{3,})',
        r'\bex\b.*?([0-9]{3,})',

        # fallback final: primer número largo
        r'([0-9]{4,})'
    ]

    for patron in patrones:
        match = re.search(patron, texto, flags=re.IGNORECASE)
        if match:
            return match.group(1)

    return None

rad_devoluciones["expediente_extraido"] = rad_devoluciones["asunto"].apply(extraer_expediente_desde_asunto)

# ------------------------------------------
# 3. SEPARAR RESPONSABLE EN NOMBRE Y REGIONAL
# ------------------------------------------
def separar_responsable(valor):
    if pd.isna(valor):
        return pd.Series([None, None])

    texto = str(valor).strip()

    match = re.match(r'^(.*?)\[(.*?)\]\s*$', texto)
    if match:
        nombre = match.group(1).strip()
        regional = match.group(2).strip()
        return pd.Series([nombre, regional])

    return pd.Series([texto, None])

rad_devoluciones[["nombre_responsable", "regional_responsable"]] = rad_devoluciones["responsable"].apply(separar_responsable)

print("Registros filtrados por devolución:", rad_devoluciones.shape)
display(
    rad_devoluciones[
        ["tipo_asunto", "asunto", "expediente_extraido", "responsable", "nombre_responsable", "regional_responsable"]
    ].head(20)
)

Registros filtrados por devolución: (238, 27)


,tipo_asunto,asunto,expediente_extraido,responsable,nombre_responsable,regional_responsable
0,Devolucion de Acto Administrativo,observaciones al proyecto de acto administrati...,66664,Argelio David Vargas Fragoz[Dirección Regional...,Argelio David Vargas Fragoz,Dirección Regional Tequendama
23,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,106783,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena
24,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,79807,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena
27,Devolucion de Acto Administrativo,Devoución Proeycto No. 677747. Exp 58681. Elab...,58681,Nirsa Yadira Rodriguez Silva[Dirección Regiona...,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá
44,Devolucion de Acto Administrativo,Devolución proyecto de Resolución No. 669376. ...,24169,Andres Felipe Barreto Prada[Dirección Regional...,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente
89,Devolucion de Acto Administrativo,Expediente 78525. Ajustes Resolución que impon...,78525,Laura Catalina Peña Roa[Dirección Regional Sum...,Laura Catalina Peña Roa,Dirección Regional Sumapaz
90,Devolucion de Acto Administrativo,Devolución expediente 93132,93132,Neila Melixa Rodriguez Parrado[Dirección Regio...,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita
148,Devolucion de Acto Administrativo,Expediente 2454. Modifica una concesión de ag...,2454,Edwin Giovanny Castillo Rocha[Dirección Region...,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro
233,Devolucion de Acto Administrativo,DEVOLUCIÓN - DAÑO EN EL ARCHIVO - EXP 70277,70277,Sandra Marcela Castañeda Castañeda[Dirección R...,Sandra Marcela Castañeda Castañeda,Dirección Regional Bogotá – La Calera
234,Devolucion de Acto Administrativo,Exp 15461. Devolución,15461,Juan Diego Poveda Rubiano[Dirección Regional S...,Juan Diego Poveda Rubiano,Dirección Regional Sabana Centro


## Segmento 4

In [131]:
# ==========================================
# SEGMENTO 4 - TABLA FINAL BASE + RESCATE DE EXPEDIENTE
# ==========================================

import re
from IPython.display import display

# ------------------------------------------
# FUNCIÓN DE RESCATE FINAL
# ------------------------------------------
def rescatar_expediente_desde_asunto(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).strip()

    patrones_rescate = [
        # formas comunes
        r'\bexpediente\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexped\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexped\.\s*([0-9]{3,})',
        r'\bexpte\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexpte\.\s*([0-9]{3,})',
        r'\bexp\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bexp\.\s*([0-9]{3,})',
        r'\bex\b\s*[:#\-\./]?\s*([0-9]{3,})',
        r'\bex\.\s*([0-9]{3,})',

        # cuando el identificador y número vienen más separados
        r'\bexpediente\b.*?([0-9]{3,})',
        r'\bexped\b.*?([0-9]{3,})',
        r'\bexpte\b.*?([0-9]{3,})',
        r'\bexp\b.*?([0-9]{3,})',
        r'\bex\b.*?([0-9]{3,})',

        # captura cualquier número largo si ya no hubo otra opción
        r'([0-9]{4,})'
    ]

    for patron in patrones_rescate:
        match = re.search(patron, texto, flags=re.IGNORECASE)
        if match:
            return match.group(1)

    return None


# ------------------------------------------
# CONSTRUIR TABLA BASE
# ------------------------------------------
tabla_devoluciones_radicados = rad_devoluciones[
    [
        "expediente_extraido",
        "tipo_asunto",
        "asunto",
        "fecha_correo",
        "f_radicado",
        "elaboro",
        "reviso",
        "responsable",
        "nombre_responsable",
        "regional_responsable"
    ]
].copy()

# ------------------------------------------
# RESCATE FINAL PARA LOS NaN
# ------------------------------------------
mask_nan = tabla_devoluciones_radicados["expediente_extraido"].isna()

tabla_devoluciones_radicados.loc[mask_nan, "expediente_extraido"] = (
    tabla_devoluciones_radicados.loc[mask_nan, "asunto"]
    .apply(rescatar_expediente_desde_asunto)
)

print("Tabla construida:", tabla_devoluciones_radicados.shape)
print("Expedientes vacíos después del rescate:",
      tabla_devoluciones_radicados["expediente_extraido"].isna().sum())

display(tabla_devoluciones_radicados.head(30))

Tabla construida: (238, 10)
Expedientes vacíos después del rescate: 0


,expediente_extraido,tipo_asunto,asunto,fecha_correo,f_radicado,elaboro,reviso,responsable,nombre_responsable,regional_responsable
0,66664,Devolucion de Acto Administrativo,observaciones al proyecto de acto administrati...,2026-01-02 07:44:28.317,2026-01-02 07:44:28.300,Jhoanna Alexandra Montoya Muñoz,NaN,Argelio David Vargas Fragoz[Dirección Regional...,Argelio David Vargas Fragoz,Dirección Regional Tequendama
23,106783,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:27:24.473,2026-01-05 11:27:24.447,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena
24,79807,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:29:11.863,2026-01-05 11:29:11.837,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena
27,58681,Devolucion de Acto Administrativo,Devoución Proeycto No. 677747. Exp 58681. Elab...,2026-01-05 11:32:14.760,2026-01-05 11:32:14.740,Nelly Avella Sanabria,NaN,Nirsa Yadira Rodriguez Silva[Dirección Regiona...,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá
44,24169,Devolucion de Acto Administrativo,Devolución proyecto de Resolución No. 669376. ...,2026-01-06 09:52:18.097,2026-01-06 09:52:18.053,Nelly Avella Sanabria,Hector Abel Castellanos Perez / DJA,Andres Felipe Barreto Prada[Dirección Regional...,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente
89,78525,Devolucion de Acto Administrativo,Expediente 78525. Ajustes Resolución que impon...,2026-01-08 08:55:56.850,2026-01-08 08:55:56.830,Iván Mauricio Castillo Arenas,NaN,Laura Catalina Peña Roa[Dirección Regional Sum...,Laura Catalina Peña Roa,Dirección Regional Sumapaz
90,93132,Devolucion de Acto Administrativo,Devolución expediente 93132,2026-01-08 10:37:51.747,2026-01-08 10:37:51.677,Lorena Del Pilar Riaño Garcia,Andrea Polania Arteaga / DJA,Neila Melixa Rodriguez Parrado[Dirección Regio...,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita
148,2454,Devolucion de Acto Administrativo,Expediente 2454. Modifica una concesión de ag...,2026-01-09 14:21:08.240,2026-01-09 14:21:08.217,Iván Mauricio Castillo Arenas,NaN,Edwin Giovanny Castillo Rocha[Dirección Region...,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro
233,70277,Devolucion de Acto Administrativo,DEVOLUCIÓN - DAÑO EN EL ARCHIVO - EXP 70277,2026-01-13 14:50:27.597,2026-01-13 14:50:27.583,Johanna Isabel Muñoz Charry,Joaquin Fernando Sarmiento Cardenas / DJA,Sandra Marcela Castañeda Castañeda[Dirección R...,Sandra Marcela Castañeda Castañeda,Dirección Regional Bogotá – La Calera
234,15461,Devolucion de Acto Administrativo,Exp 15461. Devolución,2026-01-13 14:51:49.670,2026-01-13 14:51:49.607,Diego Fernando Forero Gonzalez,NaN,Juan Diego Poveda Rubiano[Dirección Regional S...,Juan Diego Poveda Rubiano,Dirección Regional Sabana Centro


# Segmento 5 Revision y validacion de datos

In [132]:
# ==========================================
# SEGMENTO 5 - VALIDACIÓN EXPEDIENTES
# ==========================================

df = tabla_devoluciones_radicados.copy()

# total
total = len(df)

# con dato
con_expediente = df["expediente_extraido"].notna().sum()

# vacíos
sin_expediente = df["expediente_extraido"].isna().sum()

# porcentaje
porcentaje_completos = round((con_expediente / total) * 100, 2)

# tabla tipo dinámica
resumen_expedientes = pd.DataFrame({
    "total_registros": [total],
    "con_expediente": [con_expediente],
    "sin_expediente": [sin_expediente],
    "porcentaje_completos_%": [porcentaje_completos]
}) # ==========================================
# SEGMENTO 6 - LLAVE CONTRA ACTIVIDADESACTUALES
# ==========================================

import pandas as pd
import re
from IPython.display import display

def limpiar_llave_expediente(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip()
    valor = re.sub(r"\.0$", "", valor)      # quita .0 si viene de numérico
    valor = re.sub(r"[^0-9]", "", valor)    # deja solo números
    return valor if valor != "" else None

# copia de trabajo
base_revision = tabla_devoluciones_radicados.copy()
base_actividades = df_actividades.copy()

# llaves limpias
base_revision["llave_expediente"] = base_revision["expediente_extraido"].apply(limpiar_llave_expediente)
base_actividades["llave_expediente"] = base_actividades["expediente"].apply(limpiar_llave_expediente)

# nos quedamos solo con las columnas necesarias desde ActividadesActuales
actividades_llave = base_actividades[["llave_expediente", "estado_actual"]].copy()

# por si una llave aparece repetida, dejamos la primera
actividades_llave = actividades_llave.dropna(subset=["llave_expediente"]).drop_duplicates(subset=["llave_expediente"])

# merge
base_revision = base_revision.merge(
    actividades_llave,
    on="llave_expediente",
    how="left"
)

# columnas de validación
base_revision["esta_en_actividadesactuales"] = base_revision["estado_actual"].notna().map({True: "SI", False: "NO"})
base_revision["estado_actividades"] = base_revision["estado_actual"]
base_revision["Estado_Agustin"] = base_revision["esta_en_actividadesactuales"].apply(
    lambda x: "Actividad en revision" if x == "SI" else ""
)

# quitamos columna auxiliar intermedia que ya quedó reflejada
base_revision = base_revision.drop(columns=["estado_actual"])

print("Cruce terminado:", base_revision.shape)

display(resumen_expedientes)

Cruce terminado: (238, 14)


,total_registros,con_expediente,sin_expediente,porcentaje_completos_%
0,238,238,0,100.0


# Crear llave y cruzar con ActividadesActuales

In [133]:
# ==========================================
# SEGMENTO 6 - LLAVE CONTRA ACTIVIDADESACTUALES
# ==========================================

import pandas as pd
import re
from IPython.display import display

def limpiar_llave_expediente(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip()
    valor = re.sub(r"\.0$", "", valor)      # quita .0 si viene de numérico
    valor = re.sub(r"[^0-9]", "", valor)    # deja solo números
    return valor if valor != "" else None

# copia de trabajo
base_revision = tabla_devoluciones_radicados.copy()
base_actividades = df_actividades.copy()

# llaves limpias
base_revision["llave_expediente"] = base_revision["expediente_extraido"].apply(limpiar_llave_expediente)
base_actividades["llave_expediente"] = base_actividades["expediente"].apply(limpiar_llave_expediente)

# nos quedamos solo con las columnas necesarias desde ActividadesActuales
actividades_llave = base_actividades[["llave_expediente", "estado_actual"]].copy()

# por si una llave aparece repetida, dejamos la primera
actividades_llave = actividades_llave.dropna(subset=["llave_expediente"]).drop_duplicates(subset=["llave_expediente"])

# merge
base_revision = base_revision.merge(
    actividades_llave,
    on="llave_expediente",
    how="left"
)

# columnas de validación
base_revision["esta_en_actividadesactuales"] = base_revision["estado_actual"].notna().map({True: "SI", False: "NO"})
base_revision["estado_actividades"] = base_revision["estado_actual"]
base_revision["Estado_Agustin"] = base_revision["esta_en_actividadesactuales"].apply(
    lambda x: "Actividad en revision" if x == "SI" else ""
)

# quitamos columna auxiliar intermedia que ya quedó reflejada
base_revision = base_revision.drop(columns=["estado_actual"])

print("Cruce terminado:", base_revision.shape)

Cruce terminado: (238, 14)


# Segmento 7 Dejar la base final y mostrar muestra de cómo quedó    

In [134]:
# ==========================================
# SEGMENTO 7 - BASE FINAL Y MUESTRA
# ==========================================

tabla_devoluciones_radicados = base_revision.copy()

# orden sugerido de columnas
columnas_finales = [
    "expediente_extraido",
    "llave_expediente",
    "tipo_asunto",
    "asunto",
    "fecha_correo",
    "f_radicado",
    "elaboro",
    "reviso",
    "responsable",
    "nombre_responsable",
    "regional_responsable",
    "esta_en_actividadesactuales",
    "estado_actividades",
    "Estado_Agustin"
]

# dejar solo columnas existentes en ese orden
columnas_finales = [c for c in columnas_finales if c in tabla_devoluciones_radicados.columns]
tabla_devoluciones_radicados = tabla_devoluciones_radicados[columnas_finales].copy()

print("Base final actualizada:", tabla_devoluciones_radicados.shape)
display(tabla_devoluciones_radicados.head(10))

Base final actualizada: (238, 14)


,expediente_extraido,llave_expediente,tipo_asunto,asunto,fecha_correo,f_radicado,elaboro,reviso,responsable,nombre_responsable,regional_responsable,esta_en_actividadesactuales,estado_actividades,Estado_Agustin
0,66664,66664,Devolucion de Acto Administrativo,observaciones al proyecto de acto administrati...,2026-01-02 07:44:28.317,2026-01-02 07:44:28.300,Jhoanna Alexandra Montoya Muñoz,NaN,Argelio David Vargas Fragoz[Dirección Regional...,Argelio David Vargas Fragoz,Dirección Regional Tequendama,SI,En Trámite,Actividad en revision
1,106783,106783,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:27:24.473,2026-01-05 11:27:24.447,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NO,NaN,
2,79807,79807,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:29:11.863,2026-01-05 11:29:11.837,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NO,NaN,
3,58681,58681,Devolucion de Acto Administrativo,Devoución Proeycto No. 677747. Exp 58681. Elab...,2026-01-05 11:32:14.760,2026-01-05 11:32:14.740,Nelly Avella Sanabria,NaN,Nirsa Yadira Rodriguez Silva[Dirección Regiona...,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá,NO,NaN,
4,24169,24169,Devolucion de Acto Administrativo,Devolución proyecto de Resolución No. 669376. ...,2026-01-06 09:52:18.097,2026-01-06 09:52:18.053,Nelly Avella Sanabria,Hector Abel Castellanos Perez / DJA,Andres Felipe Barreto Prada[Dirección Regional...,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente,NO,NaN,
5,78525,78525,Devolucion de Acto Administrativo,Expediente 78525. Ajustes Resolución que impon...,2026-01-08 08:55:56.850,2026-01-08 08:55:56.830,Iván Mauricio Castillo Arenas,NaN,Laura Catalina Peña Roa[Dirección Regional Sum...,Laura Catalina Peña Roa,Dirección Regional Sumapaz,NO,NaN,
6,93132,93132,Devolucion de Acto Administrativo,Devolución expediente 93132,2026-01-08 10:37:51.747,2026-01-08 10:37:51.677,Lorena Del Pilar Riaño Garcia,Andrea Polania Arteaga / DJA,Neila Melixa Rodriguez Parrado[Dirección Regio...,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita,SI,En Trámite,Actividad en revision
7,2454,2454,Devolucion de Acto Administrativo,Expediente 2454. Modifica una concesión de ag...,2026-01-09 14:21:08.240,2026-01-09 14:21:08.217,Iván Mauricio Castillo Arenas,NaN,Edwin Giovanny Castillo Rocha[Dirección Region...,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro,NO,NaN,
8,70277,70277,Devolucion de Acto Administrativo,DEVOLUCIÓN - DAÑO EN EL ARCHIVO - EXP 70277,2026-01-13 14:50:27.597,2026-01-13 14:50:27.583,Johanna Isabel Muñoz Charry,Joaquin Fernando Sarmiento Cardenas / DJA,Sandra Marcela Castañeda Castañeda[Dirección R...,Sandra Marcela Castañeda Castañeda,Dirección Regional Bogotá – La Calera,SI,En Trámite,Actividad en revision
9,15461,15461,Devolucion de Acto Administrativo,Exp 15461. Devolución,2026-01-13 14:51:49.670,2026-01-13 14:51:49.607,Diego Fernando Forero Gonzalez,NaN,Juan Diego Poveda Rubiano[Dirección Regional S...,Juan Diego Poveda Rubiano,Dirección Regional Sabana Centro,NO,NaN,


# Analisis de base firmadoos

## Segmento 8 — Crear llave contra Firmados

In [135]:
# ==========================================
# SEGMENTO 8 - LLAVE CONTRA FIRMADOS
# ==========================================

base_final = tabla_devoluciones_radicados.copy()
base_firmados = df_firmados.copy()

# limpiar llave en Firmados
base_firmados["llave_expediente"] = base_firmados["expediente"].apply(limpiar_llave_expediente)

# dejar solo llave y estado de firmados si luego lo quieres usar
firmados_llave = base_firmados[["llave_expediente"]].copy()
firmados_llave = firmados_llave.dropna(subset=["llave_expediente"]).drop_duplicates(subset=["llave_expediente"])
firmados_llave["flag_firmados"] = "SI"

# merge
base_final = base_final.merge(
    firmados_llave,
    on="llave_expediente",
    how="left"
)

# bandera firmados
base_final["esta_en_firmados"] = base_final["flag_firmados"].fillna("NO")
base_final = base_final.drop(columns=["flag_firmados"])

print("Cruce con Firmados terminado:", base_final.shape)

Cruce con Firmados terminado: (238, 15)


## Segmento 9 — Actualizar columna de base y Estado_Agustin

In [136]:
# ==========================================
# SEGMENTO 9 - BASE ENCONTRADA + ESTADO_AGUSTIN
# ==========================================

# reconstruir base en la columna existente
def definir_base_encontrada(fila):
    en_actividades = pd.notna(fila.get("estado_actividades"))
    en_firmados = fila.get("esta_en_firmados") == "SI"

    if en_actividades and en_firmados:
        return "En Ambos"
    elif en_actividades:
        return "Actividades Actuales"
    elif en_firmados:
        return "En Firmados"
    else:
        return ""

base_final["esta_en_actividadesactuales"] = base_final.apply(definir_base_encontrada, axis=1)

# actualizar Estado_Agustin
def construir_estado_agustin(fila):
    estados = []

    if pd.notna(fila.get("estado_actividades")):
        estados.append("Actividad en revision")

    if fila.get("esta_en_firmados") == "SI":
        estados.append("Atendido firmado")

    return " | ".join(estados)

base_final["Estado_Agustin"] = base_final.apply(construir_estado_agustin, axis=1)

# opcional: dejar columna de apoyo al final
columnas_finales = [
    "expediente_extraido",
    "llave_expediente",
    "tipo_asunto",
    "asunto",
    "fecha_correo",
    "f_radicado",
    "elaboro",
    "reviso",
    "responsable",
    "nombre_responsable",
    "regional_responsable",
    "esta_en_actividadesactuales",
    "estado_actividades",
    "esta_en_firmados",
    "Estado_Agustin"
]

columnas_finales = [c for c in columnas_finales if c in base_final.columns]
tabla_devoluciones_radicados = base_final[columnas_finales].copy()

print("Base actualizada:", tabla_devoluciones_radicados.shape)
display(tabla_devoluciones_radicados.head(3))

Base actualizada: (238, 15)


,expediente_extraido,llave_expediente,tipo_asunto,asunto,fecha_correo,f_radicado,elaboro,reviso,responsable,nombre_responsable,regional_responsable,esta_en_actividadesactuales,estado_actividades,esta_en_firmados,Estado_Agustin
0,66664,66664,Devolucion de Acto Administrativo,observaciones al proyecto de acto administrati...,2026-01-02 07:44:28.317,2026-01-02 07:44:28.300,Jhoanna Alexandra Montoya Muñoz,NaN,Argelio David Vargas Fragoz[Dirección Regional...,Argelio David Vargas Fragoz,Dirección Regional Tequendama,Actividades Actuales,En Trámite,NO,Actividad en revision
1,106783,106783,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:27:24.473,2026-01-05 11:27:24.447,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,En Firmados,NaN,SI,Atendido firmado
2,79807,79807,Devolucion de Acto Administrativo,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:29:11.863,2026-01-05 11:29:11.837,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis[Dirección Regional...,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,,NaN,NO,


## Limpiar Datasets para entega de excel

In [137]:
# ==========================================
# LIMPIEZA PREVIA SOLO PARA EXPORTAR A EXCEL
# ==========================================

# Base original intacta para análisis y tablas dinámicas
base_programa = tabla_devoluciones_radicados.copy()

# Base limpia solo para exportación
df_exportar = tabla_devoluciones_radicados.copy()

columnas_quitar = [
    'llave_expediente',
    'tipo_asunto',
    'responsable',
    'esta_en_actividadesactuales',
    'esta_en_firmados'
]

columnas_quitar_reales = [c for c in columnas_quitar if c in df_exportar.columns]
df_exportar = df_exportar.drop(columns=columnas_quitar_reales)

print("Columnas eliminadas para Excel:")
print(columnas_quitar_reales)
print("\nDimensión base original:", base_programa.shape)
print("Dimensión base limpia para exportar:", df_exportar.shape)

display(df_exportar.head(8))

Columnas eliminadas para Excel:
['llave_expediente', 'tipo_asunto', 'responsable', 'esta_en_actividadesactuales', 'esta_en_firmados']

Dimensión base original: (238, 15)
Dimensión base limpia para exportar: (238, 10)


,expediente_extraido,asunto,fecha_correo,f_radicado,elaboro,reviso,nombre_responsable,regional_responsable,estado_actividades,Estado_Agustin
0,66664,observaciones al proyecto de acto administrati...,2026-01-02 07:44:28.317,2026-01-02 07:44:28.300,Jhoanna Alexandra Montoya Muñoz,NaN,Argelio David Vargas Fragoz,Dirección Regional Tequendama,En Trámite,Actividad en revision
1,106783,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:27:24.473,2026-01-05 11:27:24.447,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NaN,Atendido firmado
2,79807,Observaciones al proyecto “Por medio de la cua...,2026-01-05 11:29:11.863,2026-01-05 11:29:11.837,Jhoanna Alexandra Montoya Muñoz,NaN,Maria Cristina Angel Galvis,Dirección Regional Alto Magdalena,NaN,
3,58681,Devoución Proeycto No. 677747. Exp 58681. Elab...,2026-01-05 11:32:14.760,2026-01-05 11:32:14.740,Nelly Avella Sanabria,NaN,Nirsa Yadira Rodriguez Silva,Dirección Regional Chiquinquirá,NaN,
4,24169,Devolución proyecto de Resolución No. 669376. ...,2026-01-06 09:52:18.097,2026-01-06 09:52:18.053,Nelly Avella Sanabria,Hector Abel Castellanos Perez / DJA,Andres Felipe Barreto Prada,Dirección Regional Sabana Occidente,NaN,
5,78525,Expediente 78525. Ajustes Resolución que impon...,2026-01-08 08:55:56.850,2026-01-08 08:55:56.830,Iván Mauricio Castillo Arenas,NaN,Laura Catalina Peña Roa,Dirección Regional Sumapaz,NaN,
6,93132,Devolución expediente 93132,2026-01-08 10:37:51.747,2026-01-08 10:37:51.677,Lorena Del Pilar Riaño Garcia,Andrea Polania Arteaga / DJA,Neila Melixa Rodriguez Parrado,Dirección Regional Almeidas y Guatavita,En Trámite,Actividad en revision
7,2454,Expediente 2454. Modifica una concesión de ag...,2026-01-09 14:21:08.240,2026-01-09 14:21:08.217,Iván Mauricio Castillo Arenas,NaN,Edwin Giovanny Castillo Rocha,Dirección Regional Sabana Centro,NaN,


# Segmento prueba excel

In [138]:
# ==========================================
# EXPORTACIÓN FINAL (SOLO DATASET LIMPIO)
# ==========================================

import os

ruta_salida = r"C:\Users\JULIANCHO ROBLES\Desktop\PROYECTOS GITHUB\seguimiento_metas_wil\output"

archivo_viejo = "tabla_devoluciones_radicados.xlsx"
archivo_final = "tabla_devoluciones_radicados_FINAL.xlsx"

os.makedirs(ruta_salida, exist_ok=True)

ruta_vieja = os.path.join(ruta_salida, archivo_viejo)
ruta_final = os.path.join(ruta_salida, archivo_final)

# borrar archivo viejo si existe
if os.path.exists(ruta_vieja):
    os.remove(ruta_vieja)

# exportar solo el dataset final
df_exportar.to_excel(ruta_final, index=False)

print("Archivo final generado correctamente.")
print("Ruta:", ruta_final)
print("Dimensión exportada:", df_exportar.shape)

Archivo final generado correctamente.
Ruta: C:\Users\JULIANCHO ROBLES\Desktop\PROYECTOS GITHUB\seguimiento_metas_wil\output\tabla_devoluciones_radicados_FINAL.xlsx
Dimensión exportada: (238, 10)


In [139]:
# ==========================================
# TABLAS DINÁMICAS / RESUMEN EN EL PROGRAMA
# ==========================================

import pandas as pd

df_pivot = base_programa.copy()

# -------------------------------
# NORMALIZACIONES SUAVES
# -------------------------------
if 'Estado_Agustin' in df_pivot.columns:
    df_pivot['Estado_Agustin'] = (
        df_pivot['Estado_Agustin']
        .fillna('SIN DATO')
        .replace('', 'SIN DATO')
    )

if 'esta_en_actividadesactuales' in df_pivot.columns:
    df_pivot['esta_en_actividadesactuales'] = (
        df_pivot['esta_en_actividadesactuales']
        .fillna('SIN DATO')
        .replace('', 'SIN DATO')
    )

if 'regional_responsable' in df_pivot.columns:
    df_pivot['regional_responsable'] = (
        df_pivot['regional_responsable']
        .fillna('SIN DATO')
        .astype(str)
        .str.strip()
        .replace('', 'SIN DATO')
    )

if 'estado_actividades' in df_pivot.columns:
    df_pivot['estado_actividades'] = (
        df_pivot['estado_actividades']
        .fillna('SIN DATO')
        .astype(str)
        .str.strip()
        .replace('', 'SIN DATO')
    )


# =====================================
# TABLA DINÁMICA 1: ESTADO_AGUSTIN
# =====================================
if 'Estado_Agustin' in df_pivot.columns:
    td_estado_agustin = (
        pd.pivot_table(
            df_pivot.assign(conteo=1),
            index='Estado_Agustin',
            values='conteo',
            aggfunc='sum',
            fill_value=0
        )
        .reset_index()
        .sort_values('conteo', ascending=False)
        .reset_index(drop=True)
    )
else:
    td_estado_agustin = pd.DataFrame(columns=['Estado_Agustin', 'conteo'])


# =====================================
# TABLA DINÁMICA 2: BASE ENCONTRADA
# =====================================
if 'esta_en_actividadesactuales' in df_pivot.columns:
    td_actividades = (
        pd.pivot_table(
            df_pivot.assign(conteo=1),
            index='esta_en_actividadesactuales',
            values='conteo',
            aggfunc='sum',
            fill_value=0
        )
        .reset_index()
        .sort_values('conteo', ascending=False)
        .reset_index(drop=True)
    )
else:
    td_actividades = pd.DataFrame(columns=['esta_en_actividadesactuales', 'conteo'])


# =====================================
# TABLA DINÁMICA 3: RESUMEN POR REGIONAL
# =====================================
if 'regional_responsable' in df_pivot.columns:
    td_regionales = (
        pd.pivot_table(
            df_pivot.assign(conteo=1),
            index='regional_responsable',
            values='conteo',
            aggfunc='sum',
            fill_value=0
        )
        .reset_index()
        .sort_values('conteo', ascending=False)
        .reset_index(drop=True)
    )
else:
    td_regionales = pd.DataFrame(columns=['regional_responsable', 'conteo'])


# =====================================
# TABLA DINÁMICA 4: RESUMEN POR ESTADO RELACIONADO
# =====================================
if 'estado_actividades' in df_pivot.columns:
    td_estados_relacionados = (
        pd.pivot_table(
            df_pivot.assign(conteo=1),
            index='estado_actividades',
            values='conteo',
            aggfunc='sum',
            fill_value=0
        )
        .reset_index()
        .sort_values('conteo', ascending=False)
        .reset_index(drop=True)
    )
else:
    td_estados_relacionados = pd.DataFrame(columns=['estado_actividades', 'conteo'])


# =====================================
# MOSTRAR RESULTADOS
# =====================================
print("\n==============================")
print("TABLA DINÁMICA 1 - Estado_Agustin")
print("==============================")
display(td_estado_agustin)

print("\n==============================")
print("TABLA DINÁMICA 2 - Base encontrada")
print("==============================")
display(td_actividades)

print("\n==============================")
print("TABLA DINÁMICA 3 - Resumen por regional")
print("==============================")
display(td_regionales)

print("\n==============================")
print("TABLA DINÁMICA 4 - Resumen por estado relacionado")
print("==============================")
display(td_estados_relacionados)


TABLA DINÁMICA 1 - Estado_Agustin


,Estado_Agustin,conteo
0,SIN DATO,152
1,Actividad en revision,79
2,Atendido firmado,6
3,Actividad en revision | Atendido firmado,1



TABLA DINÁMICA 2 - Base encontrada


,esta_en_actividadesactuales,conteo
0,SIN DATO,152
1,Actividades Actuales,79
2,En Firmados,6
3,En Ambos,1



TABLA DINÁMICA 3 - Resumen por regional


,regional_responsable,conteo
0,Dirección Regional Alto Magdalena,27
1,Dirección Regional Sumapaz,27
2,Dirección Regional Sabana Centro,27
3,Dirección Regional Ubaté,23
4,Dirección Regional Bogotá – La Calera,18
5,Dirección Regional Soacha,17
6,Dirección Regional Sabana Occidente,17
7,SIN DATO,13
8,Dirección Jurídica Ambiental,12
9,Dirección Regional Chiquinquirá,11



TABLA DINÁMICA 4 - Resumen por estado relacionado


,estado_actividades,conteo
0,SIN DATO,158
1,En Trámite,65
2,Seguimiento y Control,9
3,Resuelto,6
